Simulation d'un réseau de neurones 3x3

In [327]:
#!pip install pyswarm -q
#!pip install bayesian-optimization -q
#!pip install mealpy -q
#!pip install scikit-optimize -q

In [328]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import numpy as np

# Pour la recherche bayésienne
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args

# Pour PSO et GWO
import mealpy.swarm_based.PSO as PSO
from mealpy.swarm_based import GWO
from mealpy.swarm_based.GWO import OriginalGWO

from mealpy.utils.space import FloatVar
from mealpy.utils.problem import Problem

In [329]:
class ThreeByThreeNN(nn.Module):
    def __init__(self):
        super(ThreeByThreeNN, self).__init__()
        self.fc1 = nn.Linear(3, 3)  # couche 1 : 3 → 3
        self.fc2 = nn.Linear(3, 3)  # couche 2 : 3 → 3
        self.fc3 = nn.Linear(3, 3)  # couche 3 : 3 → 3

    def forward(self, x):
        x = F.relu(self.fc1(x))  # activation ReLU après couche 1
        x = F.relu(self.fc2(x))  # activation ReLU après couche 2
        x = self.fc3(x)          # pas d'activation finale (à adapter selon la tâche)
        return x


Masque pour le dropout

In [330]:
p1 = 0.05 #Note : c'est la proba de dropout donc proba de masquer le neurone zij
p2 = 0.005
p3 = 0.05

class ThreeByThreeNN_MCdropout(nn.Module):
    def __init__(self, model, p1=0.05, p2=0.005, p3=0.05):
        super().__init__()
        self.model = model
        self.p1 = p1
        self.p2 = p2
        self.p3 = p3
    
    def dropout_mask(self, x, p):
        mask = (torch.rand_like(x) > p).float() / (1 - p) ##garde x1 si proba >p1, on normalise pour que l'espérance reste constante
        return mask

    def forward(self, x):
        # Couche 1 + dropout manuel
        x1 = F.relu(self.model.fc1(x))
        mask1 = self.dropout_mask(x1, self.p1)
        x1_d = x1 * mask1 #vecteur x1 auquel on a appliqué le masque, correspond à Wi

        # Couche 2 + dropout manuel
        x2 = F.relu(self.model.fc2(x1_d))
        mask2 = self.dropout_mask(x2, self.p2)
        x2_d = x2 * mask2

        # Couche 3 + dropout manuel
        x3 = self.model.fc3(x2_d)
        mask3 = self.dropout_mask(x3, self.p3)
        x3_d = x3 * mask3

        # Affichage des masques
        #print("Masque couche 1:\n", mask1)
        #print("Masque couche 2:\n", mask2)
        #print("Masque couche 3:\n", mask3)

        return x3_d


Données

In [331]:
model_orig = ThreeByThreeNN()
model = ThreeByThreeNN_MCdropout(model_orig, p1=0.05, p2=0.005, p3=0.05)

In [332]:
# Vecteur d'entrée X et sortie attendue Y (batch de taille 1)
X = torch.randn(1, 3)  # entrée shape [1, 3]
Y = model_orig(X)
Y_hat = model(X)
N = X.shape[0]

print("X : ensemble des valeurs d'entrée :", X)
print("Y : ensemble des valeurs de sortie :", Y)
print("Y_hat : ensemble des valeurs de sortie avec dropout", Y_hat)
print("N : nombre total de données :", N)

X : ensemble des valeurs d'entrée : tensor([[0.0805, 0.7031, 0.1381]])
Y : ensemble des valeurs de sortie : tensor([[-0.2528, -0.0554,  0.1192]], grad_fn=<AddmmBackward0>)
Y_hat : ensemble des valeurs de sortie avec dropout tensor([[-0.2652, -0.0569,  0.1265]], grad_fn=<MulBackward0>)
N : nombre total de données : 1


MC Dropout : inférence

In [333]:
T = 1000 #nb de forward passes
outputs = []
model.train()  # dropout activé même en inférence

with torch.no_grad():
    for _ in range(T):
        out = model(X)
        outputs.append(out.unsqueeze(0))

outputs = torch.cat(outputs, dim=0)
mean_pred = outputs.mean(dim=0)  # moyenne sur les T passes
uncertainty = outputs.var(dim=0)  # variance = incertitude estimée

In [334]:
print("MC Dropout output (mean logits):", mean_pred)
print("MC Dropout Uncertainty :", uncertainty)

MC Dropout output (mean logits): tensor([[-0.2566, -0.0562,  0.1188]])
MC Dropout Uncertainty : tensor([[0.0027, 0.0003, 0.0008]])


Métriques à minimiser

Note : j'ai mis peu d'itérations sinon le code tourne trop longtemps mais serait plus précis

Note : plus tard, prendre en compte le bruit pour l'UA ;
Ici, qu'une seule entrée X donc c'est soit 0 soit 1, à tester avec plus de valeurs

In [335]:
def compute_ua(mean_probs, Y_true, threshold=0.7):#seuil détermine si c'est certain ou pas, cf p.12 article Enhancing
    pred_class = mean_probs.argmax(dim=1)
    confidence = mean_probs.max(dim=1).values

    correct = pred_class == Y_true
    certain = confidence >= threshold

    CC = ((correct) & (certain)).sum().item()
    IU = ((~correct) & (~certain)).sum().item()
    total = len(Y_true)

    ua = (CC + IU) / total
    return ua 

In [336]:
def compute_ece(probs, labels, n_bins=10):
    confidences, predictions = torch.max(probs, dim=1)
    accuracies = predictions.eq(labels)
    ece = torch.zeros(1)
    bin_boundaries = torch.linspace(0, 1, n_bins + 1)
    for i in range(n_bins):
        in_bin = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i+1])
        prop_in_bin = in_bin.float().mean()
        if prop_in_bin.item() > 0:
            acc_in_bin = accuracies[in_bin].float().mean()
            avg_conf = confidences[in_bin].mean()
            ece += torch.abs(avg_conf - acc_in_bin) * prop_in_bin
    return ece

In [337]:
def run_dropout_trial(p1, p2, p3, X, Y, T=1000):
    model = ThreeByThreeNN_MCdropout(ThreeByThreeNN(), p1=p1, p2=p2, p3=p3)
    model.train()
    outputs = [model(X).unsqueeze(0).detach() for _ in range(T)]
    outputs = torch.cat(outputs, dim=0)
    mean_pred = outputs.mean(dim=0)

    Y_class = torch.argmax(Y, dim=1)
    log_probs = F.log_softmax(mean_pred, dim=1)
    nll = F.nll_loss(log_probs, Y_class)

    mean_probs = torch.softmax(mean_pred, dim=1)
    Y_one_hot = F.one_hot(Y_class, num_classes=3).float()
    brier = torch.mean(torch.sum((mean_probs - Y_one_hot)**2, dim=1))
    ece = compute_ece(mean_probs, Y_class)
    ua = compute_ua(mean_probs, Y_class)

    return nll.item(), brier.item(), ece.item(), ua, (p1, p2, p3)

nll, brier, ece, ua, (p1, p2, p3) = run_dropout_trial(p1, p2, p3, X, Y)

print("Negative Log-likelihood", nll)
print("Moyenne des probas", mean_probs)
print("Classe de Y", Y_one_hot)
print("Brier score", brier)
print("Expected Calibration Error", ece)
print("Uncertain Accuracy", ua)


Negative Log-likelihood 1.4437216520309448
Moyenne des probas tensor([[0.3222, 0.3326, 0.3452]])
Classe de Y tensor([[0., 0., 1.]])
Brier score 0.9277226328849792
Expected Calibration Error 0.5436668992042542
Uncertain Accuracy 1.0


In [338]:
def combined_score(nll, brier, ece, nll_max, brier_max, ece_max):
    nll_norm = nll / nll_max if nll_max > 0 else 0
    brier_norm = brier / brier_max if brier_max > 0 else 0
    ece_norm = ece / ece_max if ece_max > 0 else 0
    return (nll_norm + brier_norm + ece_norm) / 3

Méthodes d'optimisation

In [339]:
def random_search(X, Y, n_trials=30):
    all_scores = []
    for _ in range(n_trials):
        p1, p2, p3 = random.uniform(0, 0.25), random.uniform(0, 0.25), random.uniform(0, 0.25)
        nll, brier, ece, ua, params = run_dropout_trial(p1, p2, p3, X, Y)
        all_scores.append((nll, brier, ece, params))
    nll_max = max(s[0] for s in all_scores)
    brier_max = max(s[1] for s in all_scores)
    ece_max = max(s[2] for s in all_scores)
    best = min(all_scores, key=lambda s: combined_score(*s[:3], nll_max, brier_max, ece_max))
    return best[3], combined_score(*best[:3], nll_max, brier_max, ece_max)


In [340]:
def pso_search(X, Y):
    from mealpy.utils.space import FloatVar
    from mealpy.swarm_based import PSO

    def fitness(solution):
        nll, brier, ece, ua, _ = run_dropout_trial(*solution, X, Y)
        return [nll + brier + ece]

    bounds = [
        FloatVar((0.0,), (0.25,), name="p1"),
        FloatVar((0.0,), (0.25,), name="p2"),
        FloatVar((0.0,), (0.25,), name="p3"),
    ]

    problem = {
        "obj_func": fitness,
        "bounds": bounds,
        "minmax": "min",
        "log_to": None,
    }

    model = PSO.OriginalPSO(epoch=10, pop_size=10)
    best_agent = model.solve(problem)
    best_params = best_agent.solution
    best_score = sum(fitness(best_params)) / 3
    return best_params, best_score


In [341]:
def gwo_search(X, Y):
    def fitness(solution):
        nll, brier, ece, ua, _ = run_dropout_trial(*solution, X, Y)
        return nll + brier + ece

    bounds = [
        FloatVar(0.0, 0.25),
        FloatVar(0.0, 0.25),
        FloatVar(0.0, 0.25)
    ]


    problem = Problem(
        obj_func=fitness,
        bounds=bounds,
        minmax="min",
        verbose=True,
    )

    optimizer = OriginalGWO(epoch=10, pop_size=10)
    best_agent = optimizer.solve(problem)
    # best_agent est un objet Agent contenant solution et target
    return best_agent.solution, best_agent.target


In [342]:
def bayesian_search(X, Y):
    space = [Real(0.0, 0.25, name='p1'), Real(0.0, 0.25, name='p2'), Real(0.0, 0.25, name='p3')]

    @use_named_args(space)
    def objective(p1, p2, p3):
        nll, brier, ece, ua, _ = run_dropout_trial(p1, p2, p3, X, Y)
        return (nll + brier + ece) / 3

    res = gp_minimize(objective, space, n_calls=10, random_state=42)
    return res.x, res.fun


In [343]:
def choose_optimizer():
    print("Choisissez la méthode d'optimisation :")
    print("1 - Random Search")
    print("2 - Particle Swarm Optimization (PSO)")
    print("3 - Grey Wolf Optimizer (GWO)")
    print("4 - Bayesian Optimization")
    choice = input("Entrez le numéro : ").strip()
    return int(choice)


Recherche de p optimal

In [344]:
choice = choose_optimizer()

if choice == 1:
    best_params, best_score = random_search(X, Y)
elif choice == 2:
    best_params, best_score = pso_search(X, Y)
elif choice == 3:
    best_params, best_score = gwo_search(X, Y)
elif choice == 4:
    best_params, best_score = bayesian_search(X, Y)
else:
    print("Choix invalide. Utilisation de Random Search.")
    best_params, best_score = random_search(X, Y)

print(f"\nMeilleurs params : p1={best_params[0]:.3f}, p2={best_params[1]:.3f}, p3={best_params[2]:.3f}")
print("Type de best_score :", type(best_score))
print("best_score :", best_score)

Choisissez la méthode d'optimisation :
1 - Random Search
2 - Particle Swarm Optimization (PSO)
3 - Grey Wolf Optimizer (GWO)
4 - Bayesian Optimization


2025/07/16 11:25:34 PM, INFO, mealpy.swarm_based.GWO.OriginalGWO: OriginalGWO(epoch=10, pop_size=10)
2025/07/16 11:25:36 PM, INFO, mealpy.swarm_based.GWO.OriginalGWO: >>>Problem: P, Epoch: 1, Current best: 1.3946909606456757, Global best: 1.3946909606456757, Runtime: 1.32150 seconds
2025/07/16 11:25:38 PM, INFO, mealpy.swarm_based.GWO.OriginalGWO: >>>Problem: P, Epoch: 2, Current best: 1.3946909606456757, Global best: 1.3946909606456757, Runtime: 1.27571 seconds
2025/07/16 11:25:39 PM, INFO, mealpy.swarm_based.GWO.OriginalGWO: >>>Problem: P, Epoch: 3, Current best: 1.1501448303461075, Global best: 1.1501448303461075, Runtime: 1.36579 seconds
2025/07/16 11:25:41 PM, INFO, mealpy.swarm_based.GWO.OriginalGWO: >>>Problem: P, Epoch: 4, Current best: 1.1501448303461075, Global best: 1.1501448303461075, Runtime: 1.66199 seconds
2025/07/16 11:25:42 PM, INFO, mealpy.swarm_based.GWO.OriginalGWO: >>>Problem: P, Epoch: 5, Current best: 1.1501448303461075, Global best: 1.1501448303461075, Runtime: 


Meilleurs params : p1=0.054, p2=0.210, p3=0.104
Type de best_score : <class 'mealpy.utils.target.Target'>
best_score : Objectives: [1.03902557], Fitness: 1.0390255749225616


Calcul des métriques avec les probas optimales

In [345]:
nll_opt, brier_opt, ece_opt, ua_opt, _ = run_dropout_trial(*best_params, X, Y)

print("\nMétriques finales avec probabilités optimales :")
print(f"Negative Log-likelihood (NLL): {nll_opt}")
print(f"Brier score: {brier_opt}")
print(f"Expected Calibration Error (ECE): {ece_opt}")
print(f"Uncertain Accuracy (UA): {ua_opt}")



Métriques finales avec probabilités optimales :
Negative Log-likelihood (NLL): 0.7026191353797913
Brier score: 0.38363373279571533
Expected Calibration Error (ECE): 0.5047136545181274
Uncertain Accuracy (UA): 0.0


Evoquer divergence KL à minimiser (Gal et Ghahramani)

Modèle final avec les meilleures probas

In [350]:
# Utilisation des meilleures probabilités
model_final = ThreeByThreeNN_MCdropout(ThreeByThreeNN(), *best_params)
model_final.train()

T = 1000
outputs = []
with torch.no_grad():
    for _ in range(T):
        outputs.append(model_final(X).unsqueeze(0))
outputs = torch.cat(outputs, dim=0)
mean_pred = outputs.mean(dim=0)

print("\nMC Dropout output (mean logits):", mean_pred)


MC Dropout output (mean logits): tensor([[ 0.5613, -0.3254, -0.6961]])


In [351]:
Y_final = model_final(X)
distance = torch.norm(Y_hat - Y, p=2)
print("Distance Euclidienne :", distance.item())

Distance Euclidienne : 0.014429628849029541


inclure mean square error plutôt?

In [352]:
#Remarque : la variance c'est pour la régression, sinon c'est la predictive entropy pour la classification

uncertainty = outputs.var(dim=0) #variance empirique
print("MC Dropout Uncertainty:", uncertainty) 

MC Dropout Uncertainty: tensor([[0.0483, 0.0125, 0.0776]])
